In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Loading all 8 files into Python (pandas) or Excel

In [0]:
# creating dataframes for all the csv files

customers_df = pd.read_csv('customers.csv')
inventory_df = pd.read_csv('inventory.csv')
order_items_df = pd.read_csv('order_items.csv')
orders_df = pd.read_csv('orders.csv')
products_df = pd.read_csv('products.csv')
payments_df = pd.read_csv('payments.csv')
returns_df = pd.read_csv('returns.csv')
suppliers_df = pd.read_csv('suppliers.csv')

# Validate data types

## ------------------------------ Customers.csv ------------------------------

In [0]:
customers_df.info()

* registration_date should be of type datetime.
* last_login_date should be of type datetime.

In [0]:
# changing registration_date type from customers_df to datetime
customers_df['registration_date'] = pd.to_datetime(customers_df['registration_date'], format='%d-%m-%Y')
# changing last_login_date type from customers_df to datetime
customers_df['last_login_date'] = pd.to_datetime(customers_df['last_login_date'], format='%d-%m-%Y')

In [0]:
customers_df.info()

## ------------------------------ Inventory.csv ------------------------------

In [0]:
inventory_df.info()

* last_restored_date should be of type datetime

In [0]:
# changing last_restored_date type from inventory_df to datetime
inventory_df['last_restocked_date'] = pd.to_datetime(inventory_df['last_restocked_date'], format='%d-%m-%Y')
inventory_df.info()

## ------------------------------ order_items.csv ------------------------------

In [0]:
order_items_df.info()

## ------------------------------ Orders.csv ------------------------------

In [0]:
orders_df.info()

* order_date should be of type datetime
* order_time should be of type time
* delivery_date should be of type datetime

In [0]:
# changing order_date type from orders_df to datetime
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], format='%d-%m-%Y')
# changing order_time type from orders_df to time
orders_df['order_time'] = pd.to_datetime(orders_df['order_time'], format='%H:%M:%S').dt.time
# changing delivered_date type from orders_df to datetime
orders_df['delivered_date'] = pd.to_datetime(orders_df['delivered_date'], format='%d-%m-%Y')
orders_df.info()

## ------------------------------ Payments.csv ------------------------------

In [0]:
payments_df.info()

* payment_date should be of type datetime.
* payment_time should be of type datetime.
* refund_date has every column filled with null values.

In [0]:
# payment_date column in payments_df should be of type datetime
payments_df['payment_date'] = pd.to_datetime(payments_df['payment_date'], format='%d-%m-%Y')
# payment_time column in payments_df should be of type time
payments_df['payment_time'] = pd.to_datetime(payments_df['payment_time'], format='%H:%M:%S').dt.time
payments_df.info()

## ------------------------------ Products.csv ------------------------------

In [0]:
products_df.info()

## ------------------------------ Returns.csv ------------------------------

In [0]:
returns_df.info()

* return_date should be of type datetime
* refund_date should be of type datetime

In [0]:
# return_date column from returns_df should be of type datetime
returns_df['return_date'] = pd.to_datetime(returns_df['return_date'], format='%d-%m-%Y')
# refund_date column from returns_df should be of type datetime
returns_df['refund_date'] = pd.to_datetime(returns_df['refund_date'], format='%d-%m-%Y')
returns_df.info()

## ------------------------------ Suppliers.csv ------------------------------

In [0]:
suppliers_df.info()

* created_date should be of type datetime

In [0]:
# created_date column in supplies.csv should be of type datetime
suppliers_df['created_date'] = pd.to_datetime(suppliers_df['created_date'], format='%d-%m-%Y')

In [0]:
suppliers_df.info()

# Check for outliers in final_amount - are there any orders above 5 lakhs - flag the outliers.

In [0]:
orders_df.info()

In [0]:
# Checking outliers for final_amount in orders_df that are above 5 lakhs and flagging them

orders_df[orders_df['final_amount'] > 500000].shape

In [0]:
# add a flag on orders_df to flag the above outliers
orders_df['outlier_final_amount'] = np.where(orders_df['final_amount'] > 500000, 1, 0)

orders_df[orders_df['outlier_final_amount'] == 1]

# Gross Merchandise Value

In [0]:
# KPI: Gross Merchandise Value (GMV)
# Target/Benchmark: Sum of all final_amount (all statuses)
# Why it matters: Total Business Size

total_gmv = orders_df['final_amount'].sum()

print(f"Gross Merchandise Value (GMV): {total_gmv:,.2f}")

# Net Revenue

In [0]:
# KPI: Net Revenue
# Target/Benchmark: Sum of all final_amount where status = 'Delivered'
# Why it matters: Actual Money Earned by the platform

net_revenue = orders_df[orders_df['status'] == 'Delivered']['final_amount'].sum()

print(f"Net Revenue: {net_revenue:,.2f}")

# Average Order Value (AOV)

In [0]:
# KPI: Average Order Value (AOV)
# Target/Benchmark: Net Revenue / Count of Delivered Orders
# Why it matters: Higher AOV = Customers Buying More Per Visit

delivered_orders = orders_df[orders_df['status'] == 'Delivered']
average_order_value = delivered_orders['final_amount'].sum() / delivered_orders.shape[0]

print(f"Average Order Value (AOV): {average_order_value:,.2f}")

# Cancellation Rate

In [0]:
# KPI: Cancellation Rate
# Target/Benchmark: (Cancelled Orders / Total Orders) * 100
# Why it matters: Target < 10%. Higher = Lost Revenue

cancelled_orders = orders_df[orders_df['status'] == 'Cancelled'].shape[0]
total_orders = orders_df.shape[0]
cancellation_rate = (cancelled_orders / total_orders) * 100

print(f"Cancellation Rate: {cancellation_rate:.2f}%")

# Return Rate

In [0]:
# KPI: Return Rate
# Target/Benchmark: (Returned Records / Delivered Orders) * 100
# Why it matters: Target < 8%. Higher = Product Quality Issues

returned_records = returns_df.shape[0]
delivered_orders_count = orders_df[orders_df['status'] == 'Delivered'].shape[0]
return_rate = (returned_records / delivered_orders_count) * 100

print(f"Return Rate: {return_rate:.2f}%")

# Customer Lifetime Value (CLV)

In [0]:
# KPI: Customer Lifetime Value (CLV)
# Target/Benchmark: Average Total Spent per Customer Segment
# Why it matters: Premium Segment Should be 3X Regular

# Total spent per customer (delivered orders only = actual money earned)
customer_spend = (orders_df[orders_df['status'] == 'Delivered']
                 .groupby('customer_id')['final_amount']
                 .sum()
                 .rename('delivered_total_spent'))

# Join with customer segment and compute average spend per segment
clv_by_segment = (customers_df
                  .merge(customer_spend, on='customer_id', how='left')
                  .groupby('segment')['delivered_total_spent']
                  .mean())

print("Average Total Spent per Customer Segment (CLV):")
print(clv_by_segment.round(2))

premium_clv = clv_by_segment.get('Premium', 0)
regular_clv = clv_by_segment.get('Regular', 0)

if regular_clv > 0:
    ratio = premium_clv / regular_clv
    print(f"\nPremium to Regular CLV Ratio: {ratio:.2f}x (Target: 3x)")
    if ratio >= 3:
        print("Status: Target met — Premium segment is at least 3X Regular")
    else:
        print("Status: Below target — Premium segment is not yet 3X Regular")

# Month over Month Growth

In [0]:
# KPI: Month over Month Growth
# Target/Benchmark: ((This month GMV - Last Month GMV) / Last Month GMV) * 100
# Why it matters: Shows business trajectory

# Compute monthly GMV (sum of final_amount for all orders in each month)
monthly_gmv = (orders_df
               .groupby(orders_df['order_date'].dt.to_period('M'))['final_amount']
               .sum()
               .rename('gmv'))

# Compute month over month growth %
mom_growth = ((monthly_gmv - monthly_gmv.shift(1)) / monthly_gmv.shift(1) * 100).rename('mom_growth_pct')

mom_summary = pd.concat([monthly_gmv, mom_growth], axis=1)

print("Month over Month Growth (GMV):")
print(mom_summary.round(2))

latest_month = monthly_gmv.index.max()
latest_growth = mom_growth.loc[latest_month]

print(f"\nLatest Month GMV: {monthly_gmv.loc[latest_month]:,.2f}")
print(f"MoM Growth for {latest_month}: {latest_growth:.2f}%")
if latest_growth >= 0:
    print("Status: Business is growing month over month")
else:
    print("Status: Business is declining month over month")

# Top Category Reveue Share

In [0]:
# KPI: Top Category Revenue Share
# Target/Benchmark: (Category Revenue / Total Revenue) * 100
# Why it matters: Concentration risk if one category > 60%

# Revenue per category (delivered orders only = actual money earned)
# orders_df has no product_id — category and item-level revenue (total_price)
# live in order_items_df, linked to orders via order_id
orders_with_category = orders_df[['order_id', 'status']].merge(
    order_items_df[['order_id', 'category', 'total_price']],
    on='order_id',
    how='left'
)

category_revenue = (orders_with_category[orders_with_category['status'] == 'Delivered']
                    .groupby('category')['total_price']
                    .sum()
                    .sort_values(ascending=False)
                    .rename('category_revenue'))

total_revenue = category_revenue.sum()

category_revenue_share = (category_revenue / total_revenue * 100).rename('revenue_share_pct')

summary = pd.concat([category_revenue, category_revenue_share], axis=1)

print("Revenue and Share by Category (Delivered Orders):")
print(summary.round(2))

top_category = category_revenue.index[0]
top_share = category_revenue_share.loc[top_category]

print(f"\nTop Category: {top_category}")
print(f"Top Category Revenue Share: {top_share:.2f}% (Threshold: 60%)")
if top_share > 60:
    print("Status: Concentration risk — top category exceeds 60% of revenue")
else:
    print("Status: Revenue is well diversified across categories")

# Payment Failure Rate

In [0]:
# KPI: Payment Failure Rate
# Target/Benchmark: (Failed Payments / Total Payments) * 100
# Why it matters: Target < 2%. Higher = Revenue Leakage

failed_payments = payments_df[payments_df['status'] == 'Failed'].shape[0]
total_payments = payments_df.shape[0]
payment_failure_rate = (failed_payments / total_payments) * 100

print(f"Payment Failure Rate: {payment_failure_rate:.2f}% (Target: < 2%)")
if payment_failure_rate < 2:
    print("Status: Target met — payment failure rate is below 2%")
else:
    print("Status: Above target — potential revenue leakage from failed payments")

# Inventory Fail Rate

In [0]:
# KPI: Inventory Fail Rate
# Target/Benchmark: (In Stock SKUs / Total SKUs) * 100
# Why it matters: Target > 95%. Lower = Lost Sales

# Stock levels are in inventory_df (quantity_available), not products_df
in_stock_skus = inventory_df[inventory_df['quantity_available'] > 0].shape[0]
total_skus = products_df.shape[0]
inventory_fail_rate = (in_stock_skus / total_skus) * 100

print(f"Inventory In-Stock Rate: {inventory_fail_rate:.2f}% (Target: > 95%)")
if inventory_fail_rate > 95:
    print("Status: Target met — inventory availability is above 95%")
else:
    print("Status: Below target — out-of-stock SKUs causing lost sales")